# Decision Trees and Random Forest


## Exercise: Decision Trees

We are going to use the breast cancer dataset from sklearn where the goal is to classify each sample as malignant or benign (binary classification task) based on features computed from a digitized image of a fine needle aspirate (FNA) of a breast mass.  


### Load the libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier, plot_tree
%matplotlib inline
np.random.seed(1)
plt.figure(figsize=(30,30))


### Load the data

In [ ]:
# Load data
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target


### 1. Model fitting

In this exercise you need to do the following:
- Split the data into a training and a test set using test size of 30% of the training set.

- Train a decision tree classifier to the data and visualize it.

- Make a prediction for the test set

- Evaluate the model's performance by computing the accuracy score and plotting the confusion matrix. 

#### Hints: 
Decision Trees: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier

Tree Plot: https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html

Confusion matrix plot: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Apply a decisiontree classifier to the data and visualize your decision tree
#### START YOUR SOLUTION HERE ####
# Split the data into training and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
# fit model 
model = DecisionTreeClassifier().fit(X_train, y_train)
# Plot the fitted tree
plot_tree(model, filled=True, feature_names=list(cancer.feature_names))
plt.show()

# compute predictions for test set
y_pred = model.predict(X_test)

# Compute the accuracy score
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy Score = {acc:.3f}")

# Compute the confusion matrix
conf = confusion_matrix(y_test, y_pred)

# Plot the confusion matrix
sns.heatmap(conf, 
            annot=True, 
            fmt='d', 
            cbar=False, 
            cmap="coolwarm_r", 
            linewidth = 1)
plt.title("Confusion Matrix")
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

cm_display = ConfusionMatrixDisplay(conf).plot()
#### END YOUR SOLUTION HERE ###

### Tuning tree depth with grid search CV
Tune the tree depth parameter using grid seacrh cross validation. Check out depth values between 1 and 10. 
- What is the optimal tree depth and its corresponding test accuracy score?

- Plot the tree with the optimal depth parameter.

- What is the CV accuracy for the best parameter (tree depth)?

In [ ]:
# Grid Search - tuning tree depth
from sklearn.model_selection import GridSearchCV

#### START YOUR SOLUTION HERE ####
# Define grid for the parameter to test - max_depth from 1 to 10
param_grid = {
    "max_depth": range(1, 11)
}

# Define and fit model using grid search CV with 5-fold cross validation
clf = GridSearchCV(DecisionTreeClassifier(), param_grid=param_grid, cv=5)
clf.fit(X_train, y_train)
print(f"Hyperparameters: {clf.best_params_}")
print(f"Best CV Mean Accuracy Score: {clf.best_score_:.3f}")
best_max_depth = clf.best_params_["max_depth"]

# Plot the fitted tree
plot_tree(clf.best_estimator_, filled=True, feature_names=list(cancer.feature_names))

# Print results
y_test = clf.best_estimator_.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy Score = {acc:.3f}")
#### END YOUR SOLUTION HERE ####

## Exercise: Random Forest
Now we train a random forest model to the same dataset (for the same task) using the same training test split.
- Apply a random forest classifier with 100 trees to the data.
- Compute and print the training and test accuracies and compare it to the out of bag score (hint: set `oob_score = True` in classifier).

#### Hints:
Random Forest: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

OOB: https://scikit-learn.org/stable/auto_examples/ensemble/plot_ensemble_oob.html


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay

#### END YOUR SOLUTION HERE ####
# fit model 
model = RandomForestClassifier(n_estimators=100, oob_score=True)
model.fit(X_train, y_train)

# compute predictions for the training and test sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# compute the accuracy scores (test, training and OOB)
acc_train = accuracy_score(y_train, y_train_pred)
acc_test = accuracy_score(y_test, y_test_pred)
acc_oob = model.oob_score_

# print the computed scores
print(f"Training Accuracy = {acc_train:.3f}")
print(f"Test Accuracy = {acc_test:.3f}")
print(f"OOB Accuracy = {acc_oob:.3f}")

# Compute the confusion matrix
conf = confusion_matrix(y_test, y_test_pred)

# Plot the confusion matrix 
cm_display = ConfusionMatrixDisplay(conf).plot()

#### END YOUR SOLUTION HERE ####

### Tune the number of trees parameter using grid search

Use grid search CV (5 folds) to find the best number of treees (estimators) using a grid from 100 to 1000 with a step of 100. Print the best number of trees and its corresponding test accuracy score and cross validation accuracy score.

In [ ]:
#### START YOUR SOLUTION HERE ####
# Define the grid for the number of trees
param_grid = {"n_estimators": range(100, 1001, 100)}

# Do a grid search to find the optimal number of trees
clf = GridSearchCV(RandomForestClassifier(oob_score=True), param_grid=param_grid, cv=5)
clf.fit(X_train, y_train)


In [ ]:
# print the best hyperparameter
print(f"Hyperparameters: {clf.best_params_}")

# print the training CV accuracy score
print(f"CV Mean Accuracy Score: {clf.best_score_:.3f}")

# print the test accuracy score
y_pred = clf.best_estimator_.predict(X_test)
acc_test = accuracy_score(y_test, y_pred)

print("Test Accuracy Score:", clf.score(X_test, y_test))

print(f"Test accuracy score = {acc_test}")
#### END YOUR SOLUTION HERE ####

### Importance plot
Use the permutation importance to compute the feature importances for the best model from the grid search CV. 

#### Hints:
Forest importances: https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html


In [ ]:
# retrieve the relative importance of each variable and visualize the importance plot
from sklearn.inspection import permutation_importance

#### START YOUR SOLUTION HERE ####
# get the best model from the grid search CV
best_model = clf.best_estimator_

# compute the feature importances using permutation test
result = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=2)
forest_importances = pd.Series(result.importances_mean, index=cancer.feature_names)

# plot the importances
fig, ax = plt.subplots()
forest_importances.plot.bar(yerr=result.importances_std, ax=ax)
ax.set_title("Feature importances using permutation on full model")
ax.set_ylabel("Mean accuracy decrease")
fig.tight_layout()
plt.show()

# sort them (get the indices of the sorted array to be able to apply it on the errors)
sort_index = np.argsort(forest_importances)[::-1]

# plot the importances
fig, ax = plt.subplots()
forest_importances.iloc[sort_index].plot.bar(yerr=result.importances_std[sort_index], ax=ax)
ax.set_title("Feature importances using permutation on full model")
ax.set_ylabel("Mean accuracy decrease")
fig.tight_layout()
plt.show()

#### END YOUR SOLUTION HERE ####

Below we use the attribute `feature_importances_` of random forest model selected in the grid search that quantifies the feature importance based on mean decrease in impurity. These scores, however, can be misleading for continuous and high cardinality features. 

In [ ]:
# get the feature importances from the fitted model
importances = best_model.feature_importances_

# get the standard deviations
std = np.std([tree.feature_importances_ for tree in best_model.estimators_], axis=0)

# put them in pandas series
forest_importances = pd.Series(importances, index=cancer.feature_names)

# sort them
forest_importances.sort_values(inplace=True, ascending=False)

# plot them
fig, ax = plt.subplots()
forest_importances.plot.bar(yerr=std, ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()
